### Assignment 1 (4 scores):

- Use Numpy only to construct the Logistic Regression model.
- Train and evaluating (precision, recall, f1) the Logistic Regression model using the Gradient Descent approach **to classify 0 and 1 digit images** on the [MNIST](https://github.com/cvdfoundation/mnist?tab=readme-ov-file) dataset.
- Visualize the loss function of the training process.

In [191]:
import gzip 
import idx2numpy
import numpy as np 
from tqdm import tqdm
from sklearn.metrics import classification_report

In [192]:
class CFG : 
    def __init__(self):
        self.train_img_dir = 'data/train-images-idx3-ubyte.gz'
        self.train_label_dir =  'data/train-labels-idx1-ubyte.gz'
        self.test_img_dir = 'data/t10k-images-idx3-ubyte.gz'
        self.test_label_dir = 'data/t10k-labels-idx1-ubyte.gz'

In [193]:
def convert_idx_to_numpy(file_path) : 
    with gzip.open(file_path, 'rb') as f :
        data = idx2numpy.convert_from_file(f)
    
    if data is not None : 
        print(f"SHAPE OF DATA : {data.shape}")
    
    return data


In [194]:
config = CFG()

In [195]:
# read train data
print("TRAINING SET")
train_img = convert_idx_to_numpy(config.train_img_dir)
train_label = convert_idx_to_numpy(config.train_label_dir)

# read test data
print("TEST SET")
test_img = convert_idx_to_numpy(config.test_img_dir)
test_label = convert_idx_to_numpy(config.test_label_dir)

TRAINING SET
SHAPE OF DATA : (60000, 28, 28)
SHAPE OF DATA : (60000,)
TEST SET
SHAPE OF DATA : (10000, 28, 28)
SHAPE OF DATA : (10000,)


In [196]:
# normalize data 
train_img = train_img / 255.0
test_img = test_img / 255.0

In [197]:
# reshaping train data 
train_img = train_img.reshape(train_img.shape[0], -1)
train_label = train_label.reshape(-1, 1)

# reshaping test data
test_img = test_img.reshape(test_img.shape[0], -1)
test_label = test_label.reshape(-1, 1)

In [198]:
class LogisticRegression : 
    def __init__(self, lr=1e-4, epochs=100):
        self.lr = lr 
        self.epochs = epochs 
        self.W = None 
        self.b = 0 
    def softmax(self, z): 
        return np.exp(z) / np.sum(np.exp(z), axis=0)
    
    def sigmoid(self, z) :
        return 1 / (1 + np.exp(-z))
    
    def forward(self, X): 
        if self.W is None:
            return None
        z = X @ self.W.T + self.b 
        return self.sigmoid(z)
    
    def cross_entropy_loss(self, y, y_pred) : 
        return -np.mean(y * np.log(y_pred + 1e-9) + (1 - y) * np.log(1 - y_pred + 1e-9))

    def fit(self, X, y):
        nrows, ncols = X.shape

        self.W = np.zeros((1, ncols))
        self.b = 0 

        training_bar = tqdm(range(self.epochs), desc="TRAINING")
        for epoch in training_bar : 
            y_pred = self.forward(X)

            err = y_pred - y 
            dw = (1 / nrows) * np.dot(X.T, err)
            db = (1 / nrows) * np.sum(err)

            self.W -= self.lr * dw.T
            self.b -= self.lr * db 
            
            loss = self.cross_entropy_loss(y, y_pred)
            training_bar.set_postfix(loss=loss)

    def predict(self, X):
        y_pred = self.forward(X) 
        return [1 if p > 0.5 else 0 for p in y_pred]


In [199]:
def filter_data(X, y, criterion) : 
    filted_data_index = np.where(y == criterion)[0] 
    return X[filted_data_index], y[filted_data_index]

In [200]:
TRAIN_DIGIT_0_IMG, TRAIN_DIGIT_0_LABEL = filter_data(train_img, train_label, 0)
TRAIN_DIGIT_1_IMG, TRAIN_DIGIT_1_LABEL = filter_data(train_img, train_label, 1)

TEST_DIGIT_0_IMG, TEST_DIGIT_0_LABEL = filter_data(test_img, test_label, 0)
TEST_DIGIT_1_IMG, TEST_DIGIT_1_LABEL = filter_data(test_img, test_label, 1)

In [201]:
TRAIN_DIGIT_0_1_IMG = np.vstack([TRAIN_DIGIT_0_IMG, TRAIN_DIGIT_1_IMG])
TRAIN_DIGIT_0_1_LABEL = np.vstack([TRAIN_DIGIT_0_LABEL, TRAIN_DIGIT_1_LABEL])

TEST_DIGIT_0_1_IMG = np.vstack([TEST_DIGIT_0_IMG, TEST_DIGIT_1_IMG])
TEST_DIGIT_0_1_LABEL = np.vstack([TEST_DIGIT_0_LABEL, TEST_DIGIT_1_LABEL])

In [ ]:
lg = LogisticRegression() 
lg.fit(TRAIN_DIGIT_0_1_IMG, TRAIN_DIGIT_0_1_LABEL)

TRAINING: 100%|██████████| 100/100 [00:01<00:00, 97.85it/s, loss=0.662]


In [204]:
y_pred = lg.predict(TEST_DIGIT_0_1_IMG)

In [206]:
print(classification_report(TEST_DIGIT_0_1_LABEL, y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00       980
           1       1.00      0.99      1.00      1135

    accuracy                           1.00      2115
   macro avg       1.00      1.00      1.00      2115
weighted avg       1.00      1.00      1.00      2115



### Assignment 2 (4 scores):

- Use Numpy only to construct the Softmax Regression model.
- Train and evaluating (precision, recall, f1) the Softmax Regression model using the Gradient Descent approach **to classify 10 digit images** on the [MNIST](https://github.com/cvdfoundation/mnist?tab=readme-ov-file) dataset.
- Visualize the loss function of the training process.

In [242]:
def one_hot(y) : 
    categories = np.unique(y)
    one_hot_mat = np.full((y.shape[0], len(categories)), 0)
    print("ONE HOT ENCODING MAP SHAPE : ", one_hot_mat.shape)
    for row, label in zip(one_hot_mat, y) : 
        row[label] = 1
    return one_hot_mat


In [244]:
import numpy as np
import matplotlib.pyplot as plt

class SoftmaxRegression:
    def __init__(self, lr=1e-3, epochs=100):
        self.lr = lr
        self.epochs = epochs
        self.W = None
        self.b = None
        self.loss_history = []  

    def softmax(self, z):
        z = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, X):
        z = X @ self.W.T + self.b
        return self.softmax(z)

    def cross_entropy_loss(self, y_true, y_pred):
        return -np.mean(np.sum(y_true * np.log(y_pred + 1e-9), axis=1))

    def fit(self, X, y_one_hot):
        n_samples, n_features = X.shape
        n_classes = y_one_hot.shape[1]

        self.W = np.random.randn(n_classes, n_features) * 0.01
        self.b = np.zeros((1, n_classes))

        self.loss_history = []   # reset

        bar = tqdm(range(self.epochs), desc="TRAINING")

        for _ in bar:
            y_pred = self.forward(X)

            err = y_pred - y_one_hot
            dW = (err.T @ X) / n_samples
            db = np.sum(err, axis=0, keepdims=True) / n_samples

            self.W -= self.lr * dW
            self.b -= self.lr * db

            loss = self.cross_entropy_loss(y_one_hot, y_pred)
            self.loss_history.append(loss) 

            bar.set_postfix(loss=loss)

    def predict_proba(self, X):
        return self.forward(X)

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

    def plot_learning_curve(self):
        plt.figure()
        plt.plot(self.loss_history)
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Learning Curve")
        plt.grid()
        plt.show()

In [245]:
train_label_one_hot = one_hot(train_label).astype(int)

ONE HOT ENCODING MAP SHAPE :  (60000, 10)


In [246]:
sg = SoftmaxRegression() 
sg.fit(train_img, train_label_one_hot)


TRAINING: 100%|██████████| 100/100 [00:10<00:00,  9.13it/s, loss=2.2]


In [227]:
y_pred = sg.predict(test_img)
print(classification_report(test_label, y_pred))

              precision    recall  f1-score   support

           0       0.26      0.94      0.41       980
           1       0.80      0.09      0.15      1135
           2       0.64      0.63      0.63      1032
           3       0.38      0.04      0.07      1010
           4       0.77      0.02      0.05       982
           5       0.12      0.03      0.05       892
           6       0.19      0.02      0.04       958
           7       0.24      0.72      0.36      1028
           8       0.27      0.26      0.27       974
           9       0.31      0.25      0.28      1009

    accuracy                           0.30     10000
   macro avg       0.40      0.30      0.23     10000
weighted avg       0.41      0.30      0.23     10000



### Assignment 3 (2 scores):

- Use a Machine Learning library (Scikit Learn or Skorch) to implement and evaluate the Logistic Regression on the [MNIST](https://github.com/cvdfoundation/mnist?tab=readme-ov-file) dataset.
- Use a Machine Learning library (Scikit Learn or Skorch) to implement and evaluate the Softmax Regression on the [MNIST](https://github.com/cvdfoundation/mnist?tab=readme-ov-file) dataset.